In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Frozen inversion state candidate: independent validation

Run all uses the previously locked two new contents/seeds:
white swan in a pond (20261001), blue cable car in a mountain valley (20261002).
Each uses OFF/A/B: six videos, four marked sequences and two OFF references.
The full prompts come unchanged from the earlier velocity_calibration holdout
roster; they do not overlap the state development prompts or seeds.

State encoding, carrier, key, 11 four-slice windows, 2 boundary observations,
readout, observer gain 0.5 and negative-mean-innovation scores remain frozen.
No tuning, selection, retries with substituted content, or parameter scan.
Generate all six videos through native 50-step UniPC, save actual MP4, read its
pixels, VAE encode and run the existing 50-step approximate Euler inverse.
Known prompt and original time coordinates are assumed; no temporal attacks.

Report 44 marked core windows, 88 correlated state components, four full
trajectories and separate observer on/off rankings and margins. There are two
message candidates, not 22 independent bits per video. OFF is not a positive
detection; its ranks and same-case MP4 quality are reported separately.
Preserve failures, 66 total core windows, 12 boundaries and 276 latent slices.
No automatic scientific PASS, calibrated FPR or quality threshold is introduced.

Budget: 1200 Transformer forwards total, 300 native forward steps, 300 inverse
updates, 6 VAE decodes, 6 encodes, 6 MP4 saves. Models retain separate lifetimes.
Select GPU and Run all; no GPU model-name restriction. Source archive, launcher
and child logs, result and missing rows persist under
`MyDrive/Video-WM/InversionStateHoldout/inversion_state_holdout_<UTC>`.

Only local CPU/fake and static preparation checks have run. Source SHA:
847663159f01b5486c616363ce630878ec75ac77.
Development review found visible content/composition changes versus OFF (including
dev_p1_s1 A, PSNR about 10.41 dB). This frozen holdout tests transfer to new
contents/seeds; it does not certify invisibility or acceptable quality, and no
parameters are adjusted in response.

[Development quality review](https://github.com/RICHAAARC/SC-SSTW/blob/847663159f01b5486c616363ce630878ec75ac77/experiments/wan_state_clock/INVERSION_STATE_DEVELOPMENT_QUALITY.md)


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.metadata, json, os, signal, subprocess, sys
SOURCE_COMMIT = '847663159f01b5486c616363ce630878ec75ac77'
SOURCE_URL = 'https://github.com/RICHAAARC/SC-SSTW.git'
RUN_ID = 'inversion_state_holdout_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SOURCE = Path('/content') / (RUN_ID + '_source')
subprocess.run(['git', 'init', str(SOURCE)], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'remote', 'add', 'origin', SOURCE_URL], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'fetch', '--depth', '1', 'origin', SOURCE_COMMIT], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', SOURCE_COMMIT], check=True)
if subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip() != SOURCE_COMMIT:
    raise RuntimeError('Source checkout differs from pinned SHA')
def version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None
if version('torch') != '2.11.0+cu128':
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.11.0', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'diffusers==0.40.0', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'], check=True)
subprocess.run([sys.executable, '-c', "import torch,diffusers; assert str(torch.__version__) == '2.11.0+cu128', torch.__version__; assert diffusers.__version__ == '0.40.0', diffusers.__version__"], check=True)


In [ ]:
OUTPUT = Path('/content/drive/MyDrive/Video-WM/InversionStateHoldout') / RUN_ID
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
ARCHIVE = OUTPUT.parent / (RUN_ID + '.source.zip')
subprocess.run(['git', '-C', str(SOURCE), 'archive', '--format=zip', '--output', str(ARCHIVE), SOURCE_COMMIT], check=True)
LOG = OUTPUT.parent / (RUN_ID + '.launcher.log')
command = [sys.executable, '-u', '-m', 'experiments.wan_state_clock.inversion_state_holdout_run', '--output', str(OUTPUT)]
with LOG.open('w') as log:
    process = subprocess.Popen(command, cwd=SOURCE, start_new_session=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end=''); log.write(line); log.flush()
        code = process.wait()
    except BaseException:
        try:
            os.killpg(process.pid, signal.SIGTERM); process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL); process.wait()
        except ProcessLookupError:
            pass
        raise
print('Output:', OUTPUT, 'Launcher log:', LOG, 'Source archive:', ARCHIVE)
if (OUTPUT / 'result.json').exists():
    result = json.loads((OUTPUT / 'result.json').read_text())
    print(json.dumps({'status': result['status'], 'video_denominator': result['video_denominator'], 'case_denominator': result['case_denominator'], 'fixed_calls': result['fixed_calls'], 'state_summary': result.get('state_summary'), 'actual_calls_observed': result.get('actual_calls_observed'), 'call_count_case_coverage': result.get('call_count_case_coverage'), 'cases': {k: v['status'] for k, v in result['cases'].items()}}, indent=2))
if code:
    raise subprocess.CalledProcessError(code, command)
